# Program 6 – Word2Vec: Continuous Bag of Words (CBOW)

**Objective:** Train a CBOW model for word embeddings using TensorFlow/Keras.

**Dataset:** Small custom text corpus (sentences about royalty, family, animals, fruits, technology)

**Key tasks**
1. Build an Embedding + Dense layer structure.
2. Predict the center word from its context words.
3. Visualize embeddings with PCA and t-SNE.

## Theory (for viva)

**Word embedding:** a dense vector (e.g. 50 numbers) representing a word, such that similar words have similar vectors.

**Word2Vec** (Mikolov et al., 2013) has two variants:
- **CBOW** – predict the **center** word from the surrounding **context** words. ← this program
- **Skip-Gram** – predict the **context** words from the **center** word (Program 7).

**CBOW example** (window = 2):
Sentence: *"the king rules the kingdom"* → context = [the, king, the, kingdom], target = **rules**.

**Architecture**
```
context word ids (2*window) → Embedding (vocab × d) → average of context vectors (d) → Dense(vocab, softmax) → P(center word)
```
- The **Embedding layer weight matrix** (vocab × d) *is* the learned word vectors.
- Loss: sparse categorical cross-entropy over the vocabulary.
- **Cosine similarity** $\cos(a,b) = \frac{a\cdot b}{\|a\|\|b\|}$ is used to find similar words.
- **PCA / t-SNE** reduce d-dimensional vectors to 2-D for plotting.

## Step 1 – Import libraries

In [ ]:
import keras  # Keras 3 API (runs on TensorFlow backend)
from keras import layers  # Layer classes (Dense, Dropout, Embedding, ...)
import matplotlib.pyplot as plt  # Plotting graphs
import numpy as np  # Numerical arrays
from sklearn.decomposition import PCA  # PCA for 2-D visualization
from sklearn.manifold import TSNE  # t-SNE for 2-D visualization
import tensorflow as tf  # TensorFlow (tensors, GradientTape, seeds)

tf.random.set_seed(42)                          # Reproducibility for TensorFlow
np.random.seed(42)                              # Reproducibility for NumPy

## Step 2 – Text corpus

In [ ]:
corpus = [                                                                  # Small custom corpus: list of sentences
    "the king rules the kingdom with the queen",                            # Royalty theme
    "the queen is the wife of the king",                                    # Royalty theme
    "the prince is the son of the king and the queen",                      # Royalty theme
    "the princess is the daughter of the king and the queen",               # Royalty theme
    "a man can become a king and a woman can become a queen",               # Royalty + gender
    "the king and the prince ride horses in the kingdom",                   # Royalty theme
    "the queen and the princess live in the royal palace",                  # Royalty theme
    "the royal palace belongs to the king",                                 # Royalty theme
    "the man is the father and the woman is the mother",                    # Family theme
    "the boy is a young man and the girl is a young woman",                 # Family theme
    "the father and the mother love their son and daughter",                # Family theme
    "the dog chases the cat in the garden",                                 # Animals theme
    "the cat drinks milk and the dog eats meat",                            # Animals theme
    "a dog is a loyal animal and a cat is a clever animal",                 # Animals theme
    "the horse and the cow are farm animals",                               # Animals theme
    "the cow gives milk and the horse pulls the cart",                      # Animals theme
    "the puppy is a young dog and the kitten is a young cat",               # Animals theme
    "apple and banana are sweet fruits",                                    # Fruits theme
    "mango is the king of fruits",                                          # Fruits theme
    "she eats an apple and a banana every morning",                         # Fruits theme
    "orange and mango juice are healthy drinks",                            # Fruits theme
    "fresh fruits like apple orange and mango are healthy",                 # Fruits theme
    "the computer runs software written in python",                         # Tech theme
    "python is a popular programming language",                             # Tech theme
    "deep learning models run on a computer with a gpu",                    # Tech theme
    "neural networks learn patterns from data",                             # Tech theme
    "a programmer writes code in python on a computer",                     # Tech theme
    "the gpu makes deep learning training fast",                            # Tech theme
    "data is used to train neural networks and deep learning models",       # Tech theme
    "machine learning and deep learning need a lot of data",                # Tech theme
]
print("Number of sentences:", len(corpus))                                  # How many sentences we have

## Step 3 – Tokenize and build vocabulary

In [ ]:
tokenized = [sentence.lower().split() for sentence in corpus]               # Lowercase and split each sentence into words

all_words = sorted({w for sent in tokenized for w in sent})                 # Unique words, sorted for a fixed order
word2idx = {"<pad>": 0}                                                     # Index 0 reserved for padding
for w in all_words:                                                         # Give every real word an index from 1
    word2idx[w] = len(word2idx)                                             # Next free index
idx2word = {i: w for w, i in word2idx.items()}                              # Reverse lookup: index -> word
vocab_size = len(word2idx)                                                  # Total vocabulary size (incl. <pad>)

sequences = [[word2idx[w] for w in sent] for sent in tokenized]             # Convert each sentence to a list of indices

print("Vocabulary size:", vocab_size)                                       # Print vocab size
print("First sentence as indices:", sequences[0])                           # Example encoding

## Step 4 – Create (context → center) training pairs

For each word position we take `WINDOW` words on the left and right. Near sentence edges we fill missing context with `<pad>` (index 0); the Embedding layer masks these out.

In [ ]:
WINDOW = 2                                                                  # Number of context words on EACH side

contexts, targets = [], []                                                  # Lists for inputs (context) and labels (center)
for seq in sequences:                                                       # Loop over sentences
    for center_pos, center_word in enumerate(seq):                          # Loop over each word as the center
        context = []                                                        # Context word ids for this center
        for offset in range(-WINDOW, WINDOW + 1):                           # Offsets -2, -1, 0, 1, 2
            if offset == 0:                                                 # Skip the center word itself
                continue                                                    # Go to next offset
            pos = center_pos + offset                                       # Position of the context word
            context.append(seq[pos] if 0 <= pos < len(seq) else 0)          # Word id, or 0 (<pad>) if out of sentence
        contexts.append(context)                                            # Save context list
        targets.append(center_word)                                         # Save center word id

X_context = np.array(contexts, dtype="int32")                               # Shape (num_pairs, 2*WINDOW)
y_center = np.array(targets, dtype="int32")                                 # Shape (num_pairs,)

print("Training pairs:", X_context.shape[0])                                # Number of examples
for i in range(5):                                                          # Show 5 example pairs
    ctx_words = [idx2word[j] for j in X_context[i]]                         # Convert context ids to words
    print(ctx_words, "->", idx2word[y_center[i]])                           # context -> center

## Step 5 – Build the CBOW model (Embedding + average + Dense)

In [ ]:
EMBED_DIM = 50                                                              # Size of each word vector

cbow_model = keras.Sequential([                                             # CBOW network
    keras.Input(shape=(2 * WINDOW,), dtype="int32"),                        # Input: 4 context word ids
    layers.Embedding(input_dim=vocab_size,                                  # One row per vocabulary word
                     output_dim=EMBED_DIM,                                  # Each row is a 50-dim vector
                     mask_zero=True,                                        # Ignore <pad> (id 0) tokens
                     name="cbow_embedding"),                                # Name so we can fetch weights later
    layers.GlobalAveragePooling1D(),                                        # Average the context vectors (respects mask)
    layers.Dense(vocab_size, activation="softmax"),                         # Probability of each word being the center
])

cbow_model.compile(                                                         # Configure training
    optimizer=keras.optimizers.Adam(learning_rate=0.01),                    # Adam with slightly high lr (tiny dataset)
    loss="sparse_categorical_crossentropy",                                 # Center word given as integer id
    metrics=["accuracy"],                                                   # Track accuracy of predicting center word
)
cbow_model.summary()                                                        # Show architecture

## Step 6 – Train

In [ ]:
history = cbow_model.fit(                                                   # Train CBOW
    X_context, y_center,                                                    # Inputs: contexts, labels: center words
    epochs=200,                                                             # Many epochs because corpus is tiny
    batch_size=32,                                                          # Mini-batch size
    verbose=0,                                                              # Silent training (200 epochs is long)
)

plt.figure(figsize=(12, 4))                                                 # Figure
plt.subplot(1, 2, 1)                                                        # Left plot
plt.plot(history.history["loss"])                                           # Loss curve
plt.title("CBOW Training Loss")                                             # Title
plt.xlabel("Epoch")                                                         # X label
plt.subplot(1, 2, 2)                                                        # Right plot
plt.plot(history.history["accuracy"])                                       # Accuracy curve
plt.title("CBOW Training Accuracy")                                         # Title
plt.xlabel("Epoch")                                                         # X label
plt.show()                                                                  # Show
print("Final loss:", history.history["loss"][-1])                           # Last epoch loss

## Step 7 – Extract the learned embeddings

In [ ]:
embeddings = cbow_model.get_layer("cbow_embedding").get_weights()[0]        # Weight matrix, shape (vocab_size, EMBED_DIM)
print("Embedding matrix shape:", embeddings.shape)                          # Check shape
print("Vector for 'king' (first 10 values):", embeddings[word2idx["king"]][:10])  # Example word vector

## Step 8 – Test CBOW: predict center word from context

In [ ]:
def predict_center(context_words):                                          # Give a list of 4 context words
    ids = [word2idx.get(w, 0) for w in context_words]                       # Words -> ids (unknown -> <pad>)
    probs = cbow_model.predict(np.array([ids]), verbose=0)[0]               # Probability for every vocab word
    top = np.argsort(probs)[::-1][:3]                                       # Top-3 most likely center words
    return [(idx2word[i], round(float(probs[i]), 3)) for i in top]          # Return (word, probability) pairs

print(predict_center(["the", "king", "the", "kingdom"]))                    # Expected: 'rules'
print(predict_center(["cow", "gives", "and", "the"]))                       # Expected: 'milk'
print(predict_center(["is", "a", "programming", "language"]))               # Expected: 'popular'

## Step 9 – Find similar words using cosine similarity

In [ ]:
def most_similar(word, emb, top_n=5):                                       # Find nearest words to `word`
    norm = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)        # Normalize every vector to length 1
    vec = norm[word2idx[word]]                                              # Normalized vector of the query word
    sims = norm @ vec                                                       # Cosine similarity with every word
    order = np.argsort(sims)[::-1]                                          # Sort indices high -> low similarity
    result = []                                                             # Collect results
    for i in order:                                                         # Walk through sorted words
        if i == word2idx[word] or i == 0:                                   # Skip the word itself and <pad>
            continue                                                        # Next
        result.append((idx2word[i], round(float(sims[i]), 3)))              # Save (word, similarity)
        if len(result) == top_n:                                            # Stop when we have enough
            break                                                           # Exit loop
    return result                                                           # List of similar words

for w in ["king", "queen", "dog", "apple", "python"]:                       # Query words from each theme
    print(f"{w:>7} ->", most_similar(w, embeddings))                        # Print neighbours

## Step 10 – Visualize embeddings with PCA

In [ ]:
words_to_plot = [w for w in all_words if len(w) > 2]                        # Skip very short words (the, a, is ...) to reduce clutter
ids_to_plot = [word2idx[w] for w in words_to_plot]                          # Their indices
vectors = embeddings[ids_to_plot]                                           # Their embedding vectors

pca_2d = PCA(n_components=2).fit_transform(vectors)                         # Project 50-D -> 2-D with PCA

plt.figure(figsize=(14, 10))                                                # Big figure
plt.scatter(pca_2d[:, 0], pca_2d[:, 1], alpha=0.6)                          # One dot per word
for i, w in enumerate(words_to_plot):                                       # Label every dot
    plt.annotate(w, (pca_2d[i, 0], pca_2d[i, 1]), fontsize=9)               # Write word next to its point
plt.title("CBOW Word Embeddings – PCA")                                     # Title
plt.grid(True)                                                              # Grid
plt.show()                                                                  # Show

## Step 11 – Visualize embeddings with t-SNE

In [ ]:
tsne = TSNE(n_components=2,                                                 # Reduce to 2 dimensions
            perplexity=10,                                                  # Must be < number of points; small for small vocab
            random_state=42,                                                # Reproducible layout
            init="pca")                                                     # Start from PCA layout (more stable)
tsne_2d = tsne.fit_transform(vectors)                                       # Run t-SNE

plt.figure(figsize=(14, 10))                                                # Big figure
plt.scatter(tsne_2d[:, 0], tsne_2d[:, 1], alpha=0.6, color="tab:orange")    # One dot per word
for i, w in enumerate(words_to_plot):                                       # Label every dot
    plt.annotate(w, (tsne_2d[i, 0], tsne_2d[i, 1]), fontsize=9)             # Word label
plt.title("CBOW Word Embeddings – t-SNE")                                   # Title
plt.grid(True)                                                              # Grid
plt.show()                                                                  # Show

## Step 12 – Save embeddings (used for comparison in Program 7)

In [ ]:
np.save("cbow_embeddings.npy", embeddings)                                  # Save embedding matrix to disk
print("Saved cbow_embeddings.npy with shape", embeddings.shape)             # Confirm

## Expected observations
- Words from the same theme cluster together (king/queen/prince/princess; dog/cat/horse/cow; apple/mango/banana; python/computer/gpu).
- With such a tiny corpus results are noisy — real Word2Vec uses millions of sentences.

## Viva questions
1. What is the input and output of CBOW? → Input: context words; output: probability distribution of the center word.
2. Where are the word vectors stored? → In the Embedding layer's weight matrix (vocab_size × embed_dim).
3. Why average the context embeddings? → Makes the input order-independent and fixed-size (the "bag" of words).
4. Why is full softmax expensive for large vocabularies? → Output layer size = vocab size; solutions: negative sampling, hierarchical softmax.
5. PCA vs t-SNE? → PCA is linear and keeps global structure; t-SNE is non-linear and keeps local neighbourhoods (better clusters).
6. CBOW vs Skip-Gram? → CBOW is faster and better for frequent words; Skip-Gram is better for rare words and small data.